<a href="https://colab.research.google.com/github/GreaterMalrood/Forecasting-the-Energy-Transition-/blob/main/Patrick_Kane_Cap_Stone_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**BUSINESS QUESTION:**

"By modeling historical state-level price and consumption trends from the EIA SEDS dataset, how can we leverage a multi-variate predictive framework—normalized by Real State GDP (GDPRV) and adjusted by a compounding technology cost-decay factor—to pinpoint a state's 'Sustainable Crossover' and 'Fossil Fuel Inflection Point'? Furthermore, based on these long-term projections through 2500, what is the calculated deficit between a state's predicted renewable production capacity (ESTCB) and its total energy baseline demand (TETCB) that must be bridged to achieve complete economic decoupling?"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('fivethirtyeight')

from IPython.display import display

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics

from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
production = pd.read_csv("Prod_data - Sheet1.csv")
production.head(5)

In [ ]:
consumption = pd.read_csv("use_all_btu.csv")
consumption.head()

In [ ]:
energy = pd.read_csv("energy_indicators.csv")
energy.head()

In [ ]:
price = pd.read_csv("pr_all.csv")
price.head()

In [ ]:
CO2 = pd.read_csv("co2_all.csv")
CO2.head()

#**EDA**

In [ ]:
dfs = [production, consumption, energy, price, CO2]
data = pd.concat(dfs, keys=['production', 'consumption', 'energy', 'price', 'CO2'])

In [ ]:
print(data)

In [ ]:
data

In [ ]:
data.shape

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
data.dtypes

In [ ]:
data.isnull().sum()

In [ ]:
# Identify year columns in the DataFrame
prodyearcolumns = [col for col in production.columns if col.isdigit()]
conyearcolumns = [col for col in consumption.columns if col.isdigit()]
enryearcolumns = [col for col in energy.columns if col.isdigit()]
pryearcolumns = [col for col in price.columns if col.isdigit()]
CO2yearcolumns = [col for col in CO2.columns if col.isdigit()]
yearcolumns = [col for col in data.columns if col.isdigit()]

In [ ]:
# Melt the production DataFrame to transform year columns into 'Year' and 'Value'
prodmelted = production.melt(
    id_vars=['Data_Status', 'State', 'MSN'],
    value_vars=prodyearcolumns,
    var_name='Year',
    value_name='Value'
)

In [ ]:
conmelted = consumption.melt(
    id_vars=['Data_Status', 'State', 'MSN'],
    value_vars=conyearcolumns,
    var_name='Year',
    value_name='Value'
)

enrmelted = energy.melt(
    id_vars=['Data_Status', 'State', 'MSN'],
    value_vars=enryearcolumns,
    var_name='Year',
    value_name='Value'
)

prmelted = price.melt(
    id_vars=['Data_Status', 'State', 'MSN'],
    value_vars=pryearcolumns,
    var_name='Year',
    value_name='Value'
)

CO2melted = CO2.melt(
    id_vars=['Data_Status', 'State', 'MSN'],
    value_vars=CO2yearcolumns,
    var_name='Year',
    value_name='Value'
)

datamelted = data.melt(
    id_vars=['Data_Status', 'State', 'MSN'],
    value_vars=yearcolumns,
    var_name='Year',
    value_name='Value'
)


In [ ]:
prodmelted

In [ ]:
data['Total'] = data[yearcolumns].sum(axis=1)

print(data[['MSN', 'Total']].head())

In [ ]:
production['Total'] = production[prodyearcolumns].sum(axis=1)

print(production[['MSN', 'Total']].head())

In [ ]:
consumption['Total'] = consumption[conyearcolumns].sum(axis=1)

print(consumption[['MSN', 'Total']].head())

In [ ]:
energy['Total'] = energy[enryearcolumns].sum(axis=1)

print(energy[['MSN', 'Total']].head())

In [ ]:
price['Total'] = price[pryearcolumns].sum(axis=1)

print(price[['MSN', 'Total']].head())

In [ ]:
CO2['Total'] = CO2[CO2yearcolumns].sum(axis=1)

print(CO2[['MSN', 'Total']].head())

In [ ]:
data

In [ ]:
production

In [ ]:
consumption

In [ ]:
energy

In [ ]:
price

In [ ]:
CO2

In [ ]:
# Define the list of renewable energy source prefixes
# RE: Renewable Energy, WY: Wind Energy, SO: Solar Energy
# GE: Geothermal Energy, HY: Hydroelectric Power, BM: Biomass
# EA: Ethanol, B1: Biodiesel, BN: Biofuels
prefixes = ['RE', 'WY', 'SO', 'GE', 'HY', 'BM', 'EA', 'TI', 'B1', 'BN']

# Extract unique MSN codes matching the prefixes, regardless of whether they are Production
renew = [
    msn for msn in production['MSN']
    if any(msn.startswith(pre) for pre in prefixes)
]

print("Renewable MSN Variable (renew):")
print(renew)

In [ ]:
prefixes = ['RE', 'WY', 'SO', 'GE', 'HY', 'BM', 'EA', 'TI', 'B1', 'BN']

renewcon = [
    msn for msn in consumption['MSN']
    if any(msn.startswith(pre) for pre in prefixes)
]

print("Renewable MSN Variable (renew):")
print(renewcon)

In [ ]:
prefixes = ['RE', 'WY', 'SO', 'GE', 'HY', 'BM', 'EA', 'TI', 'B1', 'BN']

renewpri = [
    msn for msn in price['MSN']
    if any(msn.startswith(pre) for pre in prefixes)
]

print("Renewable MSN Variable (renew):")
print(renewpri)

In [ ]:
# Define the list of non-renewable energy source prefixes
# CL: Coal, NG: Natural Gas, NU: Nuclear, PA: Petroleum (Total),
# DF: Distillate Fuel, MG: Motor Gasoline, RF: Residual Fuel,
# AR: Asphalt, PC: Petroleum Coke, KS: Kerosene
non_renew_prefixes = ['CL', 'NG', 'NU', 'PA', 'DF', 'MG', 'RF', 'AR', 'PC', 'KS']

# Extract unique MSN codes matching the non-renewable prefixes
nonrenew = [
    msn for msn in production['MSN']
    if any(msn.startswith(pre) for pre in non_renew_prefixes)
]

print("Non-Renewable MSN Variable (non_renew):")
print(nonrenew)

In [ ]:
non_renew_prefixes = ['CL', 'NG', 'NU', 'PA', 'DF', 'MG', 'RF', 'AR', 'PC', 'KS']

nonrenewcon = [
    msn for msn in consumption['MSN']
    if any(msn.startswith(pre) for pre in non_renew_prefixes)
]

print("Non-Renewable MSN Variable (non_renew):")
print(nonrenewcon)

In [ ]:
non_renew_prefixes = ['CL', 'NG', 'NU', 'PA', 'DF', 'MG', 'RF', 'AR', 'PC', 'KS']

nonrenewpri = [
    msn for msn in price['MSN']
    if any(msn.startswith(pre) for pre in non_renew_prefixes)
]

print("Non-Renewable MSN Variable (non_renew):")
print(nonrenewpri)

In [ ]:
# Define the list of renewable energy source prefixes
# RE: Renewable Energy, WY: Wind Energy, SO: Solar Energy
# GE: Geothermal Energy, HY: Hydroelectric Power, BM: Biomass
# EA: Ethanol, B1: Biodiesel, BN: Biofuels, NU: Nuclear
prefixes = ['RE', 'WY', 'SO', 'GE', 'HY', 'BM', 'EA', 'TI', 'B1', 'BN', 'NU']

green = [
    msn for msn in production['MSN']
    if any(msn.startswith(pre) for pre in prefixes)
]

print("Renewable MSN Variable (renew):")
print(green)

In [ ]:
prefixes = ['RE', 'WY', 'SO', 'GE', 'HY', 'BM', 'EA', 'TI', 'B1', 'BN', 'NU']

congreen = [
    msn for msn in consumption['MSN']
    if any(msn.startswith(pre) for pre in prefixes)
]

print("Renewable MSN Variable (renew):")
print(congreen)

In [ ]:
prefixes = ['RE', 'WY', 'SO', 'GE', 'HY', 'BM', 'EA', 'TI', 'B1', 'BN', 'NU']

prigreen = [
    msn for msn in price['MSN']
    if any(msn.startswith(pre) for pre in prefixes)
]

print("Renewable MSN Variable (renew):")
print(prigreen)

#**HEATMAP**

In [ ]:
datanum = data.select_dtypes(include=['float64'])

corrmatrix = datanum.corr()

In [ ]:
plt.figure(figsize=(25, 20))
sns.heatmap(corrmatrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Numeric Features")
plt.show()

In [ ]:
plt.figure(figsize=(32, 24))
sns.heatmap(corrmatrix,
            annot=True,
            cmap='coolwarm',
            linewidths=0.5,
            fmt=".2f"
            )

plt.xticks(rotation=45, ha='right')
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
numproduction = production.select_dtypes(include=['float64'])
numconsumption = consumption.select_dtypes(include=['float64'])
numenergy = energy.select_dtypes(include=['float64'])
numprice = price.select_dtypes(include=['float64'])
numCO2 = CO2.select_dtypes(include=['float64'])

productionmatrix = numproduction.corr()
consumptionmatrix = numconsumption.corr()
energymatrix = numenergy.corr()
pricematrix = numprice.corr()
CO2matrix = numCO2.corr()

In [ ]:
plt.figure(figsize=(32, 24))
sns.heatmap(productionmatrix,
            annot=True,
            cmap='coolwarm',
            linewidths=0.5,
            fmt=".2f"
            )

plt.xticks(rotation=45, ha='right')
plt.title("Production Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(32, 24))
sns.heatmap(consumptionmatrix,
            annot=True,
            cmap='coolwarm',
            linewidths=0.5,
            fmt=".2f"
            )

plt.xticks(rotation=45, ha='right')
plt.title("Consumption Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(32, 24))
sns.heatmap(energymatrix,
            annot=True,
            cmap='coolwarm',
            linewidths=0.5,
            fmt=".2f"
            )

plt.xticks(rotation=45, ha='right')
plt.title("Energy Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(32, 24))
sns.heatmap(pricematrix,
            annot=True,
            cmap='coolwarm',
            linewidths=0.5,
            fmt=".2f"
            )

plt.xticks(rotation=45, ha='right')
plt.title("Price Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(32, 24))
sns.heatmap(CO2matrix,
            annot=True,
            cmap='coolwarm',
            linewidths=0.5,
            fmt=".2f"
            )

plt.xticks(rotation=45, ha='right')
plt.title("CO2 Correlation Matrix")
plt.tight_layout()
plt.show()